# HDB Town Price Growth Analysis

This notebook computes **CAGR** for each HDB town over 1, 3, and 5-year windows, anchored to **2025** (full year median prices).

It also prepares a **GeoDataFrame** by dissolving URA Master Plan 2019 planning area boundaries to match HDB town definitions, ready for choropleth mapping.

---
### Notebook Structure
1. Load & prepare resale data
2. Compute median prices per town per year (filtered to ≤ 2025)
3. Compute CAGR (1yr, 3yr, 5yr) anchored to 2025
4. Download & load URA Master Plan 2019 GeoJSON
5. Dissolve URA planning areas to HDB town boundaries
6. Join CAGR data to GeoDataFrame
7. Save outputs

## Cell 1 — Imports & Config

In [12]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import json
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Analysis config ────────────────────────────────────────────────────────
# End point anchor: last completed year
END_YEAR  = 2025
#END_QTR   = 4   # Q4
#END_LABEL = f'{END_YEAR}-Q{END_QTR}'

# CAGR windows in years
CAGR_WINDOWS = [1, 3, 5]

print(f'End point anchor : {END_YEAR}')
print(f'CAGR windows     : {CAGR_WINDOWS} years')

End point anchor : 2025
CAGR windows     : [1, 3, 5] years


## Cell 2 — Load & Prepare Resale Data

In [13]:
df = pd.read_csv(DATA_PATH)

# Parse month to datetime and extract year/quarter
df['month'] = pd.to_datetime(df['month'])
df['year']  = df['month'].dt.year
df['qtr']   = df['month'].dt.quarter
df['quarter'] = df['year'].astype(str) + '-Q' + df['qtr'].astype(str)

# Standardise town names to title case to match HDB official naming
# e.g. 'ANG MO KIO' -> 'Ang Mo Kio', 'KALLANG/WHAMPOA' -> 'Kallang/Whampoa'
df['town_clean'] = df['town'].str.title()

# Fix known edge cases after title case conversion
town_fixes = {
    'Kallang/Whampoa' : 'Kallang/ Whampoa',  # match HDB official spacing
}
df['town_clean'] = df['town_clean'].replace(town_fixes)

print(f'Loaded {len(df):,} rows')
print(f'Date range: {df["month"].min().date()} to {df["month"].max().date()}')
print(f'Unique towns: {df["town_clean"].nunique()}')
print(f'\nTowns in dataset:')
print(sorted(df['town_clean'].unique()))

Loaded 224,541 rows
Date range: 2017-01-01 to 2026-02-01
Unique towns: 26

Towns in dataset:
['Ang Mo Kio', 'Bedok', 'Bishan', 'Bukit Batok', 'Bukit Merah', 'Bukit Panjang', 'Bukit Timah', 'Central Area', 'Choa Chu Kang', 'Clementi', 'Geylang', 'Hougang', 'Jurong East', 'Jurong West', 'Kallang/ Whampoa', 'Marine Parade', 'Pasir Ris', 'Punggol', 'Queenstown', 'Sembawang', 'Sengkang', 'Serangoon', 'Tampines', 'Toa Payoh', 'Woodlands', 'Yishun']


## Cell 3 — Compute Median Price per Town per Year (filtered to ≤ 2025)

In [14]:
# Aggregate to town-year level, filtered to END_YEAR and earlier
town_qtr = (
    df[df['year'] <= END_YEAR]
    .groupby(['town_clean', 'year'])
    .agg(
        median_price  = ('resale_price', 'median'),
        n_transactions = ('resale_price', 'count')
    )
    .reset_index()
    .sort_values(['town_clean', 'year'])
)

print(f'Town-year records: {len(town_qtr):,}')
print(f'Years included: {sorted(town_qtr["year"].unique())}')

# Aggregate to town × flat_type × year level
town_flat_year = (
    df[df['year'] <= END_YEAR]
    .groupby(['town_clean', 'flat_type', 'year'])
    .agg(
        median_price   = ('resale_price', 'median'),
        n_transactions = ('resale_price', 'count')
    )
    .reset_index()
    .sort_values(['town_clean', 'flat_type', 'year'])
)

print(f'Town × flat_type × year records: {len(town_flat_year):,}')

town_qtr.head(10)

Town-year records: 234
Years included: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Town × flat_type × year records: 1,146


,town_clean,year,median_price,n_transactions
0,Ang Mo Kio,2017,360000.0,942
1,Ang Mo Kio,2018,346000.0,1014
2,Ang Mo Kio,2019,325000.0,954
3,Ang Mo Kio,2020,350000.0,997
4,Ang Mo Kio,2021,375000.0,1055
5,Ang Mo Kio,2022,425000.0,1026
6,Ang Mo Kio,2023,448888.0,1045
7,Ang Mo Kio,2024,470000.0,1060
8,Ang Mo Kio,2025,499500.0,972
9,Bedok,2017,385000.0,1148


## Cell 5 — Compute CAGR per Town (1yr, 3yr, 5yr) Anchored to 2025

In [15]:
def compute_cagr(end_price: float, start_price: float, n_years: int) -> float | None:
    """
    Compute Compound Annual Growth Rate.
    Returns None if inputs are invalid.
    """
    if pd.isna(end_price) or pd.isna(start_price) or start_price <= 0 or n_years <= 0:
        return None
    return round(((end_price / start_price) ** (1 / n_years) - 1) * 100, 2)


def get_town_median_at_year(town_df: pd.DataFrame, town: str, year: int) -> float | None:
    """
    Retrieve annual median price for a given town and year.
    Returns None if data is missing.
    """
    row = town_df[
        (town_df['town_clean'] == town) &
        (town_df['year'] == year)
    ]
    if row.empty:
        return None
    return row['median_price'].iloc[0]


def get_flat_median_at_year(town: str, flat_type: str, year: int) -> float | None:
    """
    Retrieve annual median price for a given town, flat type, and year.
    Returns None if data is missing.
    """
    row = town_flat_year[
        (town_flat_year['town_clean'] == town) &
        (town_flat_year['flat_type'] == flat_type) &
        (town_flat_year['year'] == year)
    ]
    return row['median_price'].iloc[0] if len(row) == 1 else None


# Filter town_qtr to END_YEAR and earlier is already done in Cell 3
town_qtr_filtered = town_qtr.copy()

# Build start year for each CAGR window
# e.g. 1yr: start = 2024, 3yr: 2022, 5yr: 2020
start_years = {n: END_YEAR - n for n in CAGR_WINDOWS}
print('CAGR start years:', start_years)

# --- Town-level CAGR (aggregate across all flat types) ---
towns = sorted(town_qtr_filtered['town_clean'].unique())
end_price_map = {
    town: get_town_median_at_year(town_qtr_filtered, town, END_YEAR)
    for town in towns
}

cagr_records = []
for town in towns:
    end_price = end_price_map[town]
    record = {'town': town, f'median_price_{END_YEAR}': end_price}

    for n_years, start_year in start_years.items():
        start_price = get_town_median_at_year(town_qtr_filtered, town, start_year)
        record[f'median_price_{start_year}'] = start_price
        record[f'cagr_{n_years}yr_pct']      = compute_cagr(end_price, start_price, n_years)

    cagr_records.append(record)

cagr_df = pd.DataFrame(cagr_records)

# --- National median CAGR benchmark (aggregate) ---
national_annual = (
    df[df['year'] <= END_YEAR]
    .groupby('year')
    .agg(median_price=('resale_price', 'median'))
    .reset_index()
)

def get_national_median(year: int) -> float | None:
    row = national_annual[national_annual['year'] == year]
    return row['median_price'].iloc[0] if not row.empty else None

national_end = get_national_median(END_YEAR)
national_cagr = {}
for n_years, start_year in start_years.items():
    national_start = get_national_median(start_year)
    national_cagr[f'national_cagr_{n_years}yr_pct'] = compute_cagr(national_end, national_start, n_years)

print(f'\nNational median price in {END_YEAR}: SGD {national_end:,.0f}')
print('National CAGR benchmarks:')
for k, v in national_cagr.items():
    print(f'  {k}: {v}%')

print(f'\nCAGR summary per town:')
print(cagr_df[['town', 'cagr_1yr_pct', 'cagr_3yr_pct', 'cagr_5yr_pct']].to_string(index=False))

# --- Flat-type × town CAGR ---
flat_types = sorted(df['flat_type'].unique())

cagr_by_flat_records = []
for town in towns:
    for ft in flat_types:
        record = {'town': town, 'flat_type': ft}
        end_price = get_flat_median_at_year(town, ft, END_YEAR)
        record[f'median_price_{END_YEAR}'] = end_price
        for n_years, start_year in start_years.items():
            start_price = get_flat_median_at_year(town, ft, start_year)
            record[f'median_price_{start_year}'] = start_price
            record[f'cagr_{n_years}yr_pct'] = compute_cagr(end_price, start_price, n_years)
        cagr_by_flat_records.append(record)

cagr_by_flat_df = pd.DataFrame(cagr_by_flat_records)
print(f'\nFlat-type × town CAGR rows: {len(cagr_by_flat_df):,}')
print(f'Rows with 1yr CAGR data: {cagr_by_flat_df["cagr_1yr_pct"].notna().sum():,}')

# --- National benchmarks per flat type ---
national_flat_benchmarks = {}
for ft in flat_types:
    nat_year = (
        df[(df['year'] <= END_YEAR) & (df['flat_type'] == ft)]
        .groupby('year')['resale_price'].median()
    )
    end_p = nat_year.get(END_YEAR)
    fb = {f'national_median_price_{END_YEAR}': end_p}
    for n_years, start_year in start_years.items():
        start_p = nat_year.get(start_year)
        fb[f'national_cagr_{n_years}yr_pct'] = compute_cagr(end_p, start_p, n_years)
    national_flat_benchmarks[ft] = fb

print('\nNational benchmarks by flat type:')
for ft, fb in national_flat_benchmarks.items():
    print(f'  {ft}: 1yr={fb.get("national_cagr_1yr_pct")}%, median={fb.get(f"national_median_price_{END_YEAR}"):,.0f}')

CAGR start years: {1: 2024, 3: 2022, 5: 2020}

National median price in 2025: SGD 628,000
National CAGR benchmarks:
  national_cagr_1yr_pct: 6.44%
  national_cagr_3yr_pct: 6.15%
  national_cagr_5yr_pct: 8.12%

CAGR summary per town:
            town  cagr_1yr_pct  cagr_3yr_pct  cagr_5yr_pct
      Ang Mo Kio          6.28          5.53          7.37
           Bedok          9.16          7.59          8.77
          Bishan          4.89          6.64          7.19
     Bukit Batok          4.35          7.02         10.15
     Bukit Merah          3.29          5.95          6.24
   Bukit Panjang          4.27          6.66          7.00
     Bukit Timah          7.41         10.67          6.26
    Central Area          4.07          3.62          4.76
   Choa Chu Kang          6.36          4.47          8.08
        Clementi         12.38          3.22          6.14
         Geylang         -0.87          8.36          7.89
         Hougang          3.36          6.44          7.42


## Cell 6 — Download URA Master Plan 2019 Planning Area GeoJSON

Downloads directly from data.gov.sg API and saves locally to avoid re-downloading.

In [16]:
GEOJSON_CACHE = Path('../data/ura_planning_area_2019.geojson')
URA_DATASET_ID = 'd_4765db0e87b9c86336792efe8a1f7a66'

def download_ura_geojson(dataset_id: str, cache_path: Path) -> gpd.GeoDataFrame:
    """
    Download URA Master Plan 2019 Planning Area GeoJSON from data.gov.sg.
    Uses cached file if available.
    """
    if cache_path.exists():
        print(f'Loading cached GeoJSON from {cache_path}')
        return gpd.read_file(cache_path)

    print('Fetching download URL from data.gov.sg API...')
    poll_url = f'https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download'
    response = requests.get(poll_url, timeout=30)
    response.raise_for_status()

    json_data = response.json()
    if json_data['code'] != 0:
        raise RuntimeError(f"data.gov.sg API error: {json_data['errMsg']}")

    download_url = json_data['data']['url']
    print(f'Downloading GeoJSON from {download_url}')

    geo_response = requests.get(download_url, timeout=60)
    geo_response.raise_for_status()

    cache_path.parent.mkdir(exist_ok=True)
    cache_path.write_bytes(geo_response.content)
    print(f'Saved to {cache_path}')

    return gpd.read_file(cache_path)


gdf_ura = download_ura_geojson(URA_DATASET_ID, GEOJSON_CACHE)

print(f'\nURA GeoDataFrame shape: {gdf_ura.shape}')
print(f'CRS: {gdf_ura.crs}')
print(f'\nColumns: {gdf_ura.columns.tolist()}')
print(f'\nSample planning area names:')
print(sorted(gdf_ura['PLN_AREA_N'].unique()))

Loading cached GeoJSON from ..\data\ura_planning_area_2019.geojson

URA GeoDataFrame shape: (55, 11)
CRS: EPSG:4326

Columns: ['OBJECTID', 'PLN_AREA_N', 'PLN_AREA_C', 'CA_IND', 'REGION_N', 'REGION_C', 'INC_CRC', 'FMEL_UPD_D', 'SHAPE.AREA', 'SHAPE.LEN', 'geometry']

Sample planning area names:
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BOON LAY', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL WATER CATCHMENT', 'CHANGI', 'CHANGI BAY', 'CHOA CHU KANG', 'CLEMENTI', 'DOWNTOWN CORE', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG', 'LIM CHU KANG', 'MANDAI', 'MARINA EAST', 'MARINA SOUTH', 'MARINE PARADE', 'MUSEUM', 'NEWTON', 'NORTH-EASTERN ISLANDS', 'NOVENA', 'ORCHARD', 'OUTRAM', 'PASIR RIS', 'PAYA LEBAR', 'PIONEER', 'PUNGGOL', 'QUEENSTOWN', 'RIVER VALLEY', 'ROCHOR', 'SELETAR', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'SIMPANG', 'SINGAPORE RIVER', 'SOUTHERN ISLANDS', 'STRAITS VIEW', 'SUNGEI KADUT', 'TAMPINES', 'TANGLIN', 'TENGAH', 'TOA PAYOH', 'TUAS', 'WESTERN ISLAN

## Cell 7 — Dissolve URA Planning Areas to HDB Town Boundaries

Maps each URA planning area to its corresponding HDB town, then dissolves geometries.

In [22]:
# Mapping: URA PLN_AREA_N (Title Case) -> HDB Town name
# Most are 1-to-1. Multi-area towns are explicitly listed.
URA_TO_HDB_TOWN = {
    # North
    'Sembawang'     : 'Sembawang',
    'Woodlands'     : 'Woodlands',
    'Yishun'        : 'Yishun',
    # North-East
    'Ang Mo Kio'    : 'Ang Mo Kio',
    'Hougang'       : 'Hougang',
    'Punggol'       : 'Punggol',
    'Sengkang'      : 'Sengkang',
    'Serangoon'     : 'Serangoon',
    # East
    'Bedok'         : 'Bedok',
    'Pasir Ris'     : 'Pasir Ris',
    'Tampines'      : 'Tampines',
    # West
    'Bukit Batok'   : 'Bukit Batok',
    'Bukit Panjang' : 'Bukit Panjang',
    'Choa Chu Kang' : 'Choa Chu Kang',
    'Clementi'      : 'Clementi',
    'Jurong East'   : 'Jurong East',
    'Jurong West'   : 'Jurong West',
    'Tengah'        : 'Tengah',
    # Central — single area
    'Bishan'        : 'Bishan',
    'Bukit Merah'   : 'Bukit Merah',
    'Bukit Timah'   : 'Bukit Timah',
    'Geylang'       : 'Geylang',
    'Marine Parade' : 'Marine Parade',
    'Queenstown'    : 'Queenstown',
    'Toa Payoh'     : 'Toa Payoh',
    # Central — multi-area dissolves
    'Kallang'       : 'Kallang/ Whampoa',
    'Whampoa'       : 'Kallang/ Whampoa',
    'Downtown Core' : 'Central Area',
    'Museum'        : 'Central Area',
    'Newton'        : 'Central Area',
    'Orchard'       : 'Central Area',
    'Outram'        : 'Central Area',
    'River Valley'  : 'Central Area',
    'Rochor'        : 'Central Area',
    'Singapore River': 'Central Area',
    'Straits View'  : 'Central Area',
    # Note: remaining URA planning areas (e.g. Lim Chu Kang, Western Water
    # Catchment, etc.) have no HDB resale activity and are excluded.
}

# Apply mapping to URA GeoDataFrame
# PLN_AREA_N is already in Title Case in the URA dataset
gdf_ura['hdb_town'] = gdf_ura['PLN_AREA_N'].str.title().map(URA_TO_HDB_TOWN)

# Report unmapped planning areas (expected — non-HDB areas)
unmapped = gdf_ura[gdf_ura['hdb_town'].isna()]['PLN_AREA_N'].tolist()
print(f'Unmapped URA planning areas ({len(unmapped)}) — no HDB resale activity:')
print(unmapped)

# Keep only mapped areas and dissolve to HDB town polygons
gdf_mapped = gdf_ura[gdf_ura['hdb_town'].notna()].copy()

# Ensure CRS is set (WGS84)
if gdf_mapped.crs is None:
    gdf_mapped = gdf_mapped.set_crs('EPSG:4326')

# Dissolve: merge geometries sharing the same hdb_town
gdf_towns = (
    gdf_mapped
    .dissolve(by='hdb_town', as_index=False)
    [['hdb_town', 'geometry']]
    .rename(columns={'hdb_town': 'town'})
)

print(f'\nDissolved GeoDataFrame: {len(gdf_towns)} HDB towns')
print(sorted(gdf_towns['town'].tolist()))

Unmapped URA planning areas (20) — no HDB resale activity:
['BOON LAY', 'CENTRAL WATER CATCHMENT', 'CHANGI', 'PIONEER', 'SELETAR', 'LIM CHU KANG', 'NORTH-EASTERN ISLANDS', 'NOVENA', 'SIMPANG', 'SOUTHERN ISLANDS', 'SUNGEI KADUT', 'TUAS', 'WESTERN ISLANDS', 'WESTERN WATER CATCHMENT', 'CHANGI BAY', 'MARINA EAST', 'MARINA SOUTH', 'TANGLIN', 'MANDAI', 'PAYA LEBAR']

Dissolved GeoDataFrame: 27 HDB towns
['Ang Mo Kio', 'Bedok', 'Bishan', 'Bukit Batok', 'Bukit Merah', 'Bukit Panjang', 'Bukit Timah', 'Central Area', 'Choa Chu Kang', 'Clementi', 'Geylang', 'Hougang', 'Jurong East', 'Jurong West', 'Kallang/ Whampoa', 'Marine Parade', 'Pasir Ris', 'Punggol', 'Queenstown', 'Sembawang', 'Sengkang', 'Serangoon', 'Tampines', 'Tengah', 'Toa Payoh', 'Woodlands', 'Yishun']


## Cell 8 — Join CAGR Data to GeoDataFrame

In [23]:
# Join CAGR data onto the dissolved town geometries
gdf_choropleth = gdf_towns.merge(
    cagr_df,
    left_on='town',
    right_on='town',
    how='left'
)

# Add national CAGR benchmarks as columns for easy reference in frontend
for k, v in national_cagr.items():
    gdf_choropleth[k] = v

# Flag towns above/below national CAGR for each window
for n in CAGR_WINDOWS:
    nat_val = national_cagr.get(f'national_cagr_{n}yr_pct')
    if nat_val is not None:
        gdf_choropleth[f'above_national_{n}yr'] = (
            gdf_choropleth[f'cagr_{n}yr_pct'] > nat_val
        )

# Towns with no CAGR data (e.g. Tengah — no resale transactions)
no_data_towns = gdf_choropleth[gdf_choropleth['cagr_1yr_pct'].isna()]['town'].tolist()
if no_data_towns:
    print(f'Towns with insufficient data for CAGR: {no_data_towns}')

print(f'\nFinal choropleth GeoDataFrame shape: {gdf_choropleth.shape}')
print(gdf_choropleth[['town', 'cagr_1yr_pct', 'cagr_3yr_pct', 'cagr_5yr_pct',
                       'national_cagr_1yr_pct', 'above_national_1yr']].to_string(index=False))

Towns with insufficient data for CAGR: ['Tengah']

Final choropleth GeoDataFrame shape: (27, 15)
            town  cagr_1yr_pct  cagr_3yr_pct  cagr_5yr_pct  national_cagr_1yr_pct  above_national_1yr
      Ang Mo Kio          6.28          5.53          7.37                   6.44               False
           Bedok          9.16          7.59          8.77                   6.44                True
          Bishan          4.89          6.64          7.19                   6.44               False
     Bukit Batok          4.35          7.02         10.15                   6.44               False
     Bukit Merah          3.29          5.95          6.24                   6.44               False
   Bukit Panjang          4.27          6.66          7.00                   6.44               False
     Bukit Timah          7.41         10.67          6.26                   6.44                True
    Central Area          4.07          3.62          4.76                   6.44      

## Cell 9 — Save Outputs

In [11]:
# Save 1: CAGR summary CSV (town-level aggregate)
cagr_output_path = OUTPUT_DIR / 'town_cagr_summary.csv'
cagr_df.to_csv(cagr_output_path, index=False)
print(f'Saved CAGR summary to {cagr_output_path}')

flat_cagr_path = OUTPUT_DIR / 'town_cagr_by_flat.csv'
cagr_by_flat_df.to_csv(flat_cagr_path, index=False)
print(f'Saved flat-type CAGR to {flat_cagr_path}')


Saved CAGR summary to ..\outputs\town_cagr_summary.csv
Saved flat-type CAGR to ..\outputs\town_cagr_by_flat.csv


In [ ]:
# Save 1: CAGR summary CSV (town-level aggregate)
cagr_output_path = OUTPUT_DIR / 'town_cagr_summary.csv'
cagr_df.to_csv(cagr_output_path, index=False)
print(f'Saved CAGR summary to {cagr_output_path}')


# Save 2: Choropleth GeoJSON (for frontend mapping)
choropleth_path = OUTPUT_DIR / 'town_choropleth.geojson'
gdf_choropleth.to_file(choropleth_path, driver='GeoJSON')
print(f'Saved choropleth GeoJSON to {choropleth_path}')

# Save 3: National CAGR benchmarks JSON (aggregate + per flat type)
import json
national_path = OUTPUT_DIR / 'national_cagr_benchmarks.json'
national_benchmarks = {
    **national_cagr,
    f'national_median_price_{END_YEAR}': national_end,
    'flat_type_benchmarks': national_flat_benchmarks
}
national_path.write_text(json.dumps(national_benchmarks, indent=2))
print(f'Saved national benchmarks to {national_path}')

# Save 4: Flat-type × town CAGR CSV
flat_cagr_path = OUTPUT_DIR / 'town_cagr_by_flat.csv'
cagr_by_flat_df.to_csv(flat_cagr_path, index=False)
print(f'Saved flat-type CAGR to {flat_cagr_path}')

print('\n--- All outputs saved ---')
print(f'  {cagr_output_path}')
print(f'  {choropleth_path}')
print(f'  {national_path}')
print(f'  {flat_cagr_path}')

Saved CAGR summary to ..\outputs\town_cagr_summary.csv


NameError: name 'gdf_choropleth' is not defined